### imports

In [61]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
pd.set_option('display.max_columns',200)

### datasets

In [62]:
df = pd.read_csv("india_job_market_2024_2026.csv")

### analysis

In [63]:
df.shape

df.dtypes

df.Company_Type.nunique()

4

### feature engineering


In [64]:
['Job_ID', 'Job_Title', 'Company', 'Company_Type', 'Industry', 'City',
       'Location_Tier', 'Experience_Level', 'Job_Type', 'Work_Mode',
       'Salary_LPA', 'Skills_Required', 'Education_Required', 'Openings',
       'Applicants', 'Company_Rating', 'Date_Posted']



oe_cols =  ['City','Company',
            'Job_Title','Industry',
              'Job_Type','Education_Required'       ]

one_hot_cols = ['Location_Tier','Work_Mode','Company_Type']

cols_to_drop = ['Job_ID', 
                 
                 'Salary_LPA', 'Skills_Required',  'Date_Posted',
                 ]

experience_order = [
    'Fresher (0-1 yr)', 'Junior (1-3 yrs)', 'Mid (3-6 yrs)', 
    'Senior (6-10 yrs)', 'Lead (10+ yrs)'
]


exp_encoder = OrdinalEncoder(categories=[experience_order])

oe = OrdinalEncoder()

df_encoded = pd.get_dummies(df, columns = one_hot_cols)

df_encoded[oe_cols] = oe.fit_transform(df[oe_cols])

df_encoded['Experience_Level'] = exp_encoder.fit_transform(df_encoded[['Experience_Level']])


In [72]:
X = df_encoded.drop(columns=cols_to_drop)
y = df_encoded['Salary_LPA']
X.head()

,Job_Title,Company,Industry,City,Experience_Level,Job_Type,Education_Required,Openings,Applicants,Company_Rating,Location_Tier_Remote,Location_Tier_Tier 1,Location_Tier_Tier 2,Work_Mode_Hybrid,Work_Mode_On-Site,Work_Mode_Remote,Company_Type_Indian Unicorn,Company_Type_MNC,Company_Type_PSU/Govt,Company_Type_Startup
0,1.0,47.0,9.0,16.0,3.0,1.0,4.0,3,276,4.0,True,False,False,False,False,True,False,True,False,False
1,23.0,15.0,9.0,12.0,3.0,1.0,2.0,3,325,4.0,False,False,True,True,False,False,True,False,False,False
2,4.0,21.0,4.0,16.0,3.0,1.0,6.0,5,559,3.6,True,False,False,False,False,True,False,False,True,False
3,7.0,20.0,9.0,13.0,2.0,1.0,3.0,3,184,3.5,False,True,False,True,False,False,False,False,False,True
4,22.0,38.0,4.0,16.0,1.0,1.0,6.0,1,64,3.9,True,False,False,False,False,True,False,True,False,False


### modeling


In [71]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 1)

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = RandomForestRegressor(max_leaf_nodes=max_leaf_nodes, random_state=1)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return(mae)

for max_leaf_nodes in [5, 250, 1000, 5000]:
    my_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print(f"Max leaf nodes: {max_leaf_nodes}  \t\t Mean Absolute Error: {my_mae}")

Max leaf nodes: 5  		 Mean Absolute Error: 6.402628335751939
Max leaf nodes: 250  		 Mean Absolute Error: 3.0712752135392916
Max leaf nodes: 1000  		 Mean Absolute Error: 3.097392135794044
Max leaf nodes: 5000  		 Mean Absolute Error: 3.1000904
